# Notebook 1: Data & Indexing

In [1]:
import json
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from tqdm import tqdm

/Users/arystankamalov/Desktop/Minecraft_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("naklecha/minecraft-question-answer-700k", split="train")

data = []
for i in range(5000):
    data.append({"question": ds[i]["question"], "answer": ds[i]["answer"]})

with open("data.json", "w") as f:
    json.dump(data, f)

print(len(data), "records saved")
print(data[0])

5000 records saved
{'question': 'What is the first statistic to decrease when a player performs energy-intensive actions in Minecraft?', 'answer': 'Saturation is the first statistic to decrease when a player performs energy-intensive actions, and it must be completely depleted before the visible hunger meter begins decreasing.'}


In [ ]:
text_lines = [record["answer"] for record in data]
ids = [str(i) for i in range(len(text_lines))]

client = PersistentClient(path="./db")

In [4]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def emb_text(text):
    return embedding_model.encode([text], normalize_embeddings=True).tolist()[0]

try:
    collection = client.get_collection(name="minecraft_minilm")
    print("already exists:", collection.count(), "docs")
except:
    collection = client.create_collection(name="minecraft_minilm")

    embeddings = []
    for line in tqdm(text_lines):
        embeddings.append(emb_text(line))

    collection.add(documents=text_lines, embeddings=embeddings, ids=ids)
    print("done:", collection.count(), "docs")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7357.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


already exists: 5000 docs


In [5]:
embedding_model_mpnet = SentenceTransformer("all-mpnet-base-v2")

def emb_text_mpnet(text):
    return embedding_model_mpnet.encode([text], normalize_embeddings=True).tolist()[0]

try:
    collection_mpnet = client.get_collection(name="minecraft_mpnet")
    print("already exists:", collection_mpnet.count(), "docs")
except:
    collection_mpnet = client.create_collection(name="minecraft_mpnet")

    embeddings = []
    for line in tqdm(text_lines):
        embeddings.append(emb_text_mpnet(line))

    collection_mpnet.add(documents=text_lines, embeddings=embeddings, ids=ids)
    print("done:", collection_mpnet.count(), "docs")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7343.15it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


already exists: 5000 docs


In [6]:
question = "How do I craft a sword in Minecraft?"
query_embedding = emb_text(question)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

context = "\n".join([line for line in results['documents'][0]])
print(context)

A sword can be obtained by crafting it with two iron ingots and a stick.
In Minecraft, you can craft a diamond sword by opening your crafting table and placing 3 diamonds in the top row, followed by a stick in the middle row.
The best material to use when crafting a sword in Minecraft is diamond.
